In [ ]:
%load_ext autoreload
%autoreload 2

from itertools import product
import sys
sys.path.append('/home/projects/nyosef/zvise/PixelGen/')
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D
from PixelGen.metrics import MultiModalVIMetrics
from sklearn.preprocessing import PowerTransformer
from pathlib import Path


import anndata as ad
import pixelator
import torch
import scvi
import scipy
# from scvi import autotune

import seaborn as sns
import scanpy as sc
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm

from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection

# import ray
# from ray import tune


from PixelGen.pxl_utils import train_model, get_model_latents
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA
from pixelator.common.statistics import clr_transformation, dsb_normalize


from pixelator.pna.plot import molecule_rank_plot
# from pixelator.plot import molecule_rank_plot, cell_count_plot, scatter_umi_per_upia_vs_tau
# from pixelator.statistics import c
# lr_transformation
# from pixelator.analysis.normalization import dsb_normalize


from sklearn.preprocessing import StandardScaler, MinMaxScaler 

from PixelGen.pxl_utils import train_model, get_model_latents, convert_polarization_to_feature_matrix, \
     convert_colocalization_to_feature_matrix, download_pxl
from PixelGen.scvi_utils import plot_losses, pca_neighbors_umap, calc_PCA, add_one_hot_encoding_obsm, plot_cumulative_variance
from PixelGen.common_utils import standardize, std_clip, filter_hv, split_pair_column, filter_df_by_two_columns, rank_plot
from PixelGen.metrics import MultiModalVIMetrics, distr_autocorrelation_in_latent
from PixelGen.multimodalvi import MultiModalSCVI
from PixelGen.multimodalvae import MultiModalVAE, AggMethod, D
from PixelGen.enums import AggMethod, D

import tempfile

from scvi import REGISTRY_KEYS
from scvi.module.base import (
    BaseModuleClass,
    LossOutput,
    PyroBaseModuleClass,
    auto_move_data,
)
from torch.distributions import NegativeBinomial, Normal, Poisson, MixtureSameFamily, Beta
from torch.distributions import kl_divergence as kl



# from cytovi import CytoVI

print(torch.cuda.is_available())
from sklearn.decomposition import PCA

from anndata import AnnData
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
sc.set_figure_params(figsize=(6, 6), frameon=False)

sns.set_theme()
torch.set_float32_matmul_precision("high")
save_dir = tempfile.TemporaryDirectory()

from utils import plot_latent, plot_gene_heatmap, plot_model_latents,get_dense,calculate_metrics,plot_composite_ppc
%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
import glob

In [ ]:
adata=sc.read_h5ad('/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/cache/adata_after_celltype_annotation.h5ad')

GRAPHS_DIR = "/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/GVAE/GRAPHS"
GVAE_NODE_MODEL='/home/projects/nyosef/zvise/PixelGen/PixelGen/PBMSC/GVAE/models/gvae_gat_model.pth'
files = glob.glob(f"{GRAPHS_DIR}/*.pt")

In [ ]:
import torch
import torch.nn as nn
from torch_geometric.nn import VGAE, GATv2Conv
from torch.utils.checkpoint import checkpoint 
import os

# --- 1. CONFIGURATION (Must match training exactly) ---
INPUT_DIM = 158       
HIDDEN_DIM = 64       
LATENT_DIM = 16       
HEADS = 4             # Important: Must match training
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = GVAE_NODE_MODEL  # Make sure this variable is defined

# --- 2. DEFINE ARCHITECTURE ---
# This class definition must be identical to the training script
class VariationalGATEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=HEADS, concat=True)
        hidden_out = hidden_channels * HEADS
        self.conv_mu = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)
        self.conv_logstd = GATv2Conv(hidden_out, out_channels, heads=1, concat=False)

    def run_conv1(self, x, edge_index):
        return self.conv1(x, edge_index).relu()

    def forward(self, x, edge_index):
        # We can disable checkpointing for inference/loading to keep it simple
        # unless you are continuing training on low memory
        if self.training: 
            x.requires_grad_(True) 
            x = checkpoint(self.run_conv1, x, edge_index, use_reentrant=False)
        else:
            x = self.run_conv1(x, edge_index)
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

# --- 3. INITIALIZE & LOAD ---
print("🔄 Initializing model...")

try:
    # 1. Instantiate the model structure
    encoder = VariationalGATEncoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM)
    model = VGAE(encoder)
    
    # 2. Load the state dictionary
    # map_location handles CPU/GPU mismatch
    state_dict = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
    
    # 3. Apply weights
    model.load_state_dict(state_dict)
    
    # 4. Move to device
    model.to(DEVICE)
    model.eval() # Set to evaluation mode by default
    
    print(f"✅ Model loaded successfully from: {MODEL_PATH}")

except RuntimeError as e:
    print(f"❌ Error loading model: {e}")
    print("Tip: Check if INPUT_DIM (158) or HEADS (4) match the trained model.")

In [ ]:
model

In [ ]:
import torch
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import os
from tqdm import tqdm

# --- 1. CONFIGURATION ---
TARGET_MARKER = "CD44"          # The protein to analyze
TARGET_CELL_GROUP = "CD8"       # Only look at CD8 T-cells
MAX_MOLECULES = 150             # Downsample per cell to avoid overcrowding
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Helper to get marker index
# Assuming adata.var_names contains the protein names in order
try:
    MARKER_IDX = list(adata.var_names).index(TARGET_MARKER)
    print(f"Target '{TARGET_MARKER}' found at index {MARKER_IDX}")
except ValueError:
    raise ValueError(f"Marker {TARGET_MARKER} not found in adata.var_names")

# --- 2. EXTRACTION LOOP ---
molecule_vectors = []
origin_cell_ids = []

# Filter files (Optimization: don't load B-cells if we only want CD8)
if "cell_type" in adata.obs:
    valid_cells = set(adata.obs[adata.obs["cell_type"] == TARGET_CELL_GROUP].index)
    filtered_files = [f for f in files if os.path.basename(f).replace(".pt", "") in valid_cells]
else:
    filtered_files = files # Fallback if no cell_type column
    print("⚠️ Warning: No 'cell_type' column found. Processing all cells.")

print(f"Processing {len(filtered_files)} cells...")
model.eval()

with torch.no_grad():
    for file_path in tqdm(filtered_files):
        try:
            # Load Data
            data = torch.load(file_path, weights_only=False).to(DEVICE)
            cell_id = os.path.basename(file_path).replace(".pt", "")
            
            # Get Latent Embeddings (Shape: [Num_Nodes, 16])
            z_all = model.encode(data.x, data.edge_index)
            
            # Identify Nodes of the Target Protein
            # data.x argmax gives the protein index (0..158)
            node_ids = data.x.argmax(dim=1)
            is_target = (node_ids == MARKER_IDX)
            
            z_target = z_all[is_target]
            
            # Skip empty cells
            if z_target.shape[0] == 0:
                continue
                
            # Downsample if too many molecules
            if z_target.shape[0] > MAX_MOLECULES:
                perm = torch.randperm(z_target.shape[0])[:MAX_MOLECULES]
                z_target = z_target[perm]
                
            # Collect
            molecule_vectors.append(z_target.cpu().numpy())
            origin_cell_ids.extend([cell_id] * z_target.shape[0])
            
        except Exception as e:
            continue

# Stack Matrix
X_molecules = np.vstack(molecule_vectors)
print(f"\n✅ Extracted {X_molecules.shape[0]} molecules of {TARGET_MARKER}")

# --- 3. CREATE "MOLECULE" ANNDATA ---
# We treat every individual protein molecule as an observation
adata_mol = ad.AnnData(X=X_molecules)
adata_mol.obs['origin_cell'] = origin_cell_ids

# Map Metadata (Condition) from the Parent Cell
# We look up what condition the 'origin_cell' belongs to
id_to_cond = adata.obs['condition'].to_dict()
adata_mol.obs['condition'] = adata_mol.obs['origin_cell'].map(id_to_cond)

# --- 4. VISUALIZATION ---
print("Running UMAP on molecule embeddings...")

# We use the raw 16-dim latent vectors directly (no need for PCA on 16 dims)
sc.pp.neighbors(adata_mol, use_rep='X', n_neighbors=30, metric='cosine')
sc.tl.umap(adata_mol)

# Plotting
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sc.pl.umap(
    adata_mol, 
    color=['condition'], 
    title=f"Spatial State of {TARGET_MARKER} (in {TARGET_CELL_GROUP})",
    palette='viridis', 
    alpha=0.6,
    size=10
)

In [ ]:
sc.pl.umap(
    adata_mol, 
    color=['condition'], 
    title=f"Spatial State of {TARGET_MARKER} (in {TARGET_CELL_GROUP})",
    palette='viridis', 
    alpha=0.6,
    size=10
)

In [ ]:
import torch
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import os
from tqdm import tqdm

# --- 1. CONFIGURATION ---
TARGET_MARKER = "CD54"          # The protein to analyze
TARGET_CELL_GROUP = "CD8"       # Only look at CD8 T-cells
MAX_MOLECULES = 150             # Downsample per cell to avoid overcrowding
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Helper to get marker index
# Assuming adata.var_names contains the protein names in order
try:
    MARKER_IDX = list(adata.var_names).index(TARGET_MARKER)
    print(f"Target '{TARGET_MARKER}' found at index {MARKER_IDX}")
except ValueError:
    raise ValueError(f"Marker {TARGET_MARKER} not found in adata.var_names")

# --- 2. EXTRACTION LOOP ---
molecule_vectors = []
origin_cell_ids = []

# Filter files (Optimization: don't load B-cells if we only want CD8)
if "cell_type" in adata.obs:
    valid_cells = set(adata.obs[adata.obs["cell_type"] == TARGET_CELL_GROUP].index)
    filtered_files = [f for f in files if os.path.basename(f).replace(".pt", "") in valid_cells]
else:
    filtered_files = files # Fallback if no cell_type column
    print("⚠️ Warning: No 'cell_type' column found. Processing all cells.")

print(f"Processing {len(filtered_files)} cells...")
model.eval()

with torch.no_grad():
    for file_path in tqdm(filtered_files):
        try:
            # Load Data
            data = torch.load(file_path, weights_only=False).to(DEVICE)
            cell_id = os.path.basename(file_path).replace(".pt", "")
            
            # Get Latent Embeddings (Shape: [Num_Nodes, 16])
            z_all = model.encode(data.x, data.edge_index)
            
            # Identify Nodes of the Target Protein
            # data.x argmax gives the protein index (0..158)
            node_ids = data.x.argmax(dim=1)
            is_target = (node_ids == MARKER_IDX)
            
            z_target = z_all[is_target]
            
            # Skip empty cells
            if z_target.shape[0] == 0:
                continue
                
            # Downsample if too many molecules
            if z_target.shape[0] > MAX_MOLECULES:
                perm = torch.randperm(z_target.shape[0])[:MAX_MOLECULES]
                z_target = z_target[perm]
                
            # Collect
            molecule_vectors.append(z_target.cpu().numpy())
            origin_cell_ids.extend([cell_id] * z_target.shape[0])
            
        except Exception as e:
            continue

# Stack Matrix
X_molecules = np.vstack(molecule_vectors)
print(f"\n✅ Extracted {X_molecules.shape[0]} molecules of {TARGET_MARKER}")

# --- 3. CREATE "MOLECULE" ANNDATA ---
# We treat every individual protein molecule as an observation
adata_mol = ad.AnnData(X=X_molecules)
adata_mol.obs['origin_cell'] = origin_cell_ids

# Map Metadata (Condition) from the Parent Cell
# We look up what condition the 'origin_cell' belongs to
id_to_cond = adata.obs['condition'].to_dict()
adata_mol.obs['condition'] = adata_mol.obs['origin_cell'].map(id_to_cond)

# --- 4. VISUALIZATION ---
print("Running UMAP on molecule embeddings...")

# We use the raw 16-dim latent vectors directly (no need for PCA on 16 dims)
sc.pp.neighbors(adata_mol, use_rep='X', n_neighbors=30, metric='cosine')
sc.tl.umap(adata_mol)

# Plotting
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sc.pl.umap(
    adata_mol, 
    color=['condition'], 
    title=f"Spatial State of {TARGET_MARKER} (in {TARGET_CELL_GROUP})",
    palette='viridis', 
    alpha=0.6,
    size=10
)

In [ ]:
adata_mol.obs.condition.value_counts()

In [ ]:
sc.pl.umap(
    adata_mol, 
    color=['condition'], 
    title=f"Spatial State of {TARGET_MARKER} (in {TARGET_CELL_GROUP})",
    palette='viridis', 
    alpha=0.6,
    size=10
)